# vad_card

> VAD chunk card renderer for the alignment card stack

In [ ]:
#| default_exp components.vad_card

In [ ]:
#| export
from typing import Any, Callable, Optional, Set

from fasthtml.common import Button, Div, Span

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles
from cjm_fasthtml_daisyui.components.data_display.card import card_body
from cjm_fasthtml_daisyui.components.feedback.loading import loading, loading_styles, loading_sizes
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_sizes, btn_colors, btn_behaviors, btn_modifiers
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui

from cjm_fasthtml_design_system.text_tiers import text_tiers

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, font_family, text_nowrap
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap, grow, shrink
)
from cjm_fasthtml_tailwind.utilities.layout import position, right, top, display_tw
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.transforms import translate
from cjm_fasthtml_tailwind.utilities.effects import opacity
from cjm_fasthtml_tailwind.utilities.transitions_and_animation import transition, duration
from cjm_fasthtml_tailwind.core.base import combine_classes

# Icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Card stack library
from cjm_fasthtml_card_stack.core.constants import CardRole
from cjm_fasthtml_card_stack.core.models import CardRenderContext

# Design system recipes (V10 P5 content_card, V11 icon-size roles)
from cjm_fasthtml_design_system.panels import panels
from cjm_fasthtml_design_system.icons import icons

# Local imports
from cjm_transcript_vad_align.html_ids import AlignmentHtmlIds
from cjm_transcript_vad_align.models import VADChunk
from cjm_transcript_vad_align.utils import format_time_precise

## Card Renderer

Each VAD chunk card is a compact row showing time range and duration.
Much simpler than segment cards — no split mode, no token selector.

In [ ]:
#| export
def render_vad_card(
    chunk:VADChunk,  # VAD chunk to render
    card_role:CardRole,  # Role of this card in viewport ("focused" or "context")
    has_boundary_above:bool=False,  # Audio file boundary exists above this card
    has_boundary_below:bool=False,  # Audio file boundary exists below this card
) -> Any:  # VAD chunk card component
    """Render a single VAD chunk card with time range, duration, playing indicator, and play button."""
    is_focused = card_role == "focused"
    is_context = card_role == "context"

    # Time range display
    time_range = Span(
        f"{format_time_precise(chunk.start_time)} → {format_time_precise(chunk.end_time)}",
        cls=combine_classes(
            font_size.sm, font_family.mono, text_nowrap,
            text_dui.base_content if is_focused else text_tiers.secondary
        )
    )

    # Duration badge
    duration_badge = Span(
        f"{chunk.duration:.1f}s",
        cls=combine_classes(badge, badge_styles.ghost, font_size.xs, font_family.mono)
    )

    # Playing indicator — hidden by default, toggled visible by JS during audio playback
    playing_indicator = Div(
        Span(cls=combine_classes(loading, loading_styles.bars, loading_sizes.xs, text_dui.secondary)),
        cls=combine_classes(
            "vad-playing-indicator",
        ),
        style="visibility:hidden;",
    )

    # Play button — active on focused card, disabled on context cards
    # type="button" prevents form submission inside StepFlow's <form> wrapper
    play_icon = lucide_icon("play", size=icons.compact_button)
    if is_focused:
        play_btn = Button(
            play_icon,
            type="button",
            cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary, btn_modifiers.circle),
            onclick="if(window.replayAlignSegment) window.replayAlignSegment();",
        )
    else:
        play_btn = Button(
            play_icon,
            type="button",
            cls=combine_classes(btn, btn_sizes.xs, btn_behaviors.disabled, btn_modifiers.circle),
            tabindex="-1",
        )

    # Boundary borders only on non-focused cards
    boundary_cls = ""
    if is_context:
        if has_boundary_above:
            boundary_cls = combine_classes(border.t(4), border_dui.neutral)
        if has_boundary_below:
            boundary_cls = combine_classes(boundary_cls, border.b(4), border_dui.neutral)

    return Div(
        Div(
            # Index badge
            Span(
                f"#{chunk.index + 1}",
                cls=combine_classes(
                    font_size.xs, font_family.mono, font_weight.bold,
                    w(10), shrink(0), opacity(50)
                )
            ),

            # Time range
            time_range,

            # Playing indicator
            playing_indicator,

            # Spacer
            Div(cls=str(grow())),

            # Duration badge
            duration_badge,

            # Play button
            play_btn,

            cls=combine_classes(
                card_body, p(3),
                flex_display, flex_direction.row, items.center, gap(3)
            )
        ),

        id=AlignmentHtmlIds.vad_chunk(chunk.index),
        cls=combine_classes(
            panels.content_card, "vad-card",
            position.relative,
            w.full,
            transition.all, duration(150),
            boundary_cls,
        ),
        data_chunk_index=str(chunk.index),
        data_audio_file_index=str(chunk.audio_file_index),
        data_start_time=str(chunk.start_time),
        data_end_time=str(chunk.end_time),
        data_card_role=card_role
    )

## Card Renderer Factory

Creates a callback compatible with the card stack library's `render_card` parameter.

In [ ]:
#| export
def create_vad_card_renderer(
    audio_file_boundaries:Set[int]=None,  # Indices where audio_file_index changes
) -> Callable:  # Card renderer callback: (item, CardRenderContext) -> FT
    """Create a card renderer callback for VAD chunk cards."""
    boundaries = audio_file_boundaries or set()

    def _render(
        item:Any,  # VADChunk instance
        context:CardRenderContext,  # Render context from card stack library
    ) -> Any:  # Rendered VAD chunk card component
        """Render a VAD chunk card for the given item and viewport context."""
        idx = context.index
        return render_vad_card(
            chunk=item,
            card_role=context.card_role,
            has_boundary_above=(idx in boundaries),
            has_boundary_below=((idx + 1) in boundaries),
        )
    return _render

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()